In [19]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [20]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import (
    load_diabetes,
    load_breast_cancer,
    load_digits,
    load_iris
)

from sklearn.model_selection import train_test_split

In [21]:
%pip install numpy pandas matplotlib scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [22]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import (
    load_diabetes,
    load_breast_cancer,
    load_digits,
    load_iris
)

from sklearn.model_selection import train_test_split

print("All libraries imported successfully!")

All libraries imported successfully!


In [23]:
# Load Diabetes Dataset
diabetes = load_diabetes(scaled=False)

X_diabetes = diabetes.data
y_diabetes = diabetes.target


# Load Breast Cancer Dataset
cancer = load_breast_cancer()

X_cancer = cancer.data
y_cancer = 1 - cancer.target


# Load Digits Dataset
digits = load_digits()

X_digits = digits.data / 16.0
y_digits = digits.target


# Load Iris Dataset
iris = load_iris()

X_iris = iris.data
y_iris = iris.target

In [24]:
print("Diabetes:", X_diabetes.shape, y_diabetes.shape)
print("Breast Cancer:", X_cancer.shape, y_cancer.shape)
print("Digits:", X_digits.shape, y_digits.shape)
print("Iris:", X_iris.shape, y_iris.shape)

Diabetes: (442, 10) (442,)
Breast Cancer: (569, 30) (569,)
Digits: (1797, 64) (1797,)
Iris: (150, 4) (150,)


In [25]:
def split_data(X, y, is_classification=True):

    stratify_value = y if is_classification else None

    # First split: 80% training + validation, 20% testing
    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=stratify_value
    )

    stratify_train = y_train_val if is_classification else None

    # Second split: 75% training, 25% validation
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val, y_train_val,
        test_size=0.25,
        random_state=42,
        stratify=stratify_train
    )

    return X_train, X_val, X_test, y_train, y_val, y_test

In [26]:
datasets = {
    "Diabetes": (X_diabetes, y_diabetes, False),
    "Breast Cancer": (X_cancer, y_cancer, True),
    "Digits": (X_digits, y_digits, True),
    "Iris": (X_iris, y_iris, True)
}

splits = {}

for name, (X, y, is_classification) in datasets.items():

    splits[name] = split_data(X, y, is_classification)

    print(f"\n{name}")

    X_train, X_val, X_test, y_train, y_val, y_test = splits[name]

    print("Training:", len(X_train))
    print("Validation:", len(X_val))
    print("Testing:", len(X_test))

    if is_classification:

        print("Class counts:")

        for split_name, labels in [
            ("Train", y_train),
            ("Validation", y_val),
            ("Test", y_test)
        ]:

            classes, counts = np.unique(labels, return_counts=True)

            print(split_name, dict(zip(classes, counts)))


Diabetes
Training: 264
Validation: 89
Testing: 89

Breast Cancer
Training: 341
Validation: 114
Testing: 114
Class counts:
Train {0: 214, 1: 127}
Validation {0: 71, 1: 43}
Test {0: 72, 1: 42}

Digits
Training: 1077
Validation: 360
Testing: 360
Class counts:
Train {0: 106, 1: 109, 2: 107, 3: 109, 4: 109, 5: 109, 6: 109, 7: 107, 8: 104, 9: 108}
Validation {0: 36, 1: 37, 2: 35, 3: 37, 4: 36, 5: 36, 6: 36, 7: 36, 8: 35, 9: 36}
Test {0: 36, 1: 36, 2: 35, 3: 37, 4: 36, 5: 37, 6: 36, 7: 36, 8: 35, 9: 36}

Iris
Training: 90
Validation: 30
Testing: 30
Class counts:
Train {0: 30, 1: 30, 2: 30}
Validation {0: 10, 1: 10, 2: 10}
Test {0: 10, 1: 10, 2: 10}


Part C - Logistic Regression

In [27]:
# Standardization class
class Standardizer:

    def fit(self, X):
        self.mean = np.mean(X, axis=0)
        self.std = np.std(X, axis=0)

        # Avoid division by zero
        self.std[self.std == 0] = 1

    def transform(self, X):
        return (X - self.mean) / self.std


# Add bias column
def add_bias(X):
    ones = np.ones((X.shape[0], 1))
    return np.concatenate((ones, X), axis=1)

In [28]:
# Extract Breast Cancer splits from Part A

X_cancer_train, X_cancer_val, X_cancer_test, y_cancer_train, y_cancer_val, y_cancer_test = splits["Breast Cancer"]

print("Before standardization:")
print("Training:", X_cancer_train.shape)
print("Validation:", X_cancer_val.shape)
print("Testing:", X_cancer_test.shape)

# Standardize using training data only
cancer_scaler = Standardizer()

cancer_scaler.fit(X_cancer_train)

X_cancer_train_std = cancer_scaler.transform(X_cancer_train)
X_cancer_val_std = cancer_scaler.transform(X_cancer_val)
X_cancer_test_std = cancer_scaler.transform(X_cancer_test)

# Add bias column
X_cancer_train_bias = add_bias(X_cancer_train_std)
X_cancer_val_bias = add_bias(X_cancer_val_std)
X_cancer_test_bias = add_bias(X_cancer_test_std)

print("\nAfter standardization and adding bias:")
print("Training:", X_cancer_train_bias.shape)
print("Validation:", X_cancer_val_bias.shape)
print("Testing:", X_cancer_test_bias.shape)

print("\nMean of standardized training features:")
print(np.round(X_cancer_train_std.mean(axis=0)[:5], 6))


Before standardization:
Training: (341, 30)
Validation: (114, 30)
Testing: (114, 30)

After standardization and adding bias:
Training: (341, 31)
Validation: (114, 31)
Testing: (114, 31)

Mean of standardized training features:
[ 0.  0. -0.  0. -0.]


In [30]:
def sigmoid(z):
    z = np.asarray(z, dtype=float)

    result = np.empty_like(z)

    positive = z >= 0

    result[positive] = 1 / (1 + np.exp(-z[positive]))

    exp_z = np.exp(z[~positive])

    result[~positive] = exp_z / (1 + exp_z)

    return result

In [31]:
test_values = np.array([-800, -2, 0, 2, 800])

print(sigmoid(test_values))

[0.         0.11920292 0.5        0.88079708 1.        ]


In [32]:
def logistic_loss(X, y, w, lam):

    z = X @ w

    data_loss = np.mean(np.logaddexp(0, z) - y * z)

    regularization = (lam / 2) * np.sum(w[1:] ** 2)

    return data_loss + regularization

In [33]:
w_test = np.zeros(X_cancer_train_bias.shape[1])

loss = logistic_loss(
    X_cancer_train_bias,
    y_cancer_train,
    w_test,
    lam=0.1
)

print("Initial Loss:", loss)

Initial Loss: 0.6931471805599453


In [34]:
def logistic_gradient(X, y, w, lam):

    z = X @ w

    p = sigmoid(z)

    grad = X.T @ (p - y) / len(y)

    grad[1:] += lam * w[1:]

    return grad

In [35]:
w_test = np.zeros(X_cancer_train_bias.shape[1])

grad = logistic_gradient(
    X_cancer_train_bias,
    y_cancer_train,
    w_test,
    lam=0.1
)

print("Gradient shape:", grad.shape)
print("First 5 gradient values:", grad[:5])

Gradient shape: (31,)
First 5 gradient values: [ 0.12756598 -0.35615821 -0.20851311 -0.36150822 -0.34543143]
